In [13]:
import pandas as pd

Выполняем данный скрипт вторым, указывая объединённый файл с продажами

In [14]:
df = pd.read_excel(r"C:\Users\m.olshanskiy\PycharmProjects\ndv_parsing\НашДомРФ\Продажи\Продажи_1ый_Химкинский.xlsx", dtype={'Корпус': str})

In [15]:
# адрес итогового файла
output_path = r"C:\Users\m.olshanskiy\Desktop\Первый Химкинский\Продажи_1ый_Химкинский.xlsx"

In [16]:
group_cols = ['Название проекта', 'Месяц_год', '№ проектной декларации']

result_parts = []

for _, group in df.groupby(group_cols):

    # если в группе больше одного корпуса → это дубликаты
    if group['Id дом.рф'].nunique() > 1:

        new_row = group.iloc[[0]].copy()

        # объединяем корпуса
        корпуса = sorted(group['Корпус'].astype(str).unique())
        new_row['Корпус'] = ', '.join(корпуса)

        # объединяем Id дом.рф
        ids = sorted(group['Id дом.рф'].astype(str).unique())
        new_row['Id дом.рф'] = ', '.join(ids)

        result_parts.append(new_row)

    else:
        # если корпус один — ничего не меняем
        result_parts.append(group)

df1 = pd.concat(result_parts, ignore_index=True)

In [17]:
group_cols = ['Название проекта', 'Месяц_год']

sum_cols = [
    'Жилые помещения, количество договоров',
    'Жилые помещения, площадь объектов',
    'Жилые помещения, суммарная цена договоров',
    'Нежилые помещения, количество договоров',
    'Нежилые помещения, площадь объектов',
    'Нежилые помещения, суммарная цена договоров',
    'Машино-места, количество договоров',
    'Машино-места, площадь объектов',
    'Машино-места, суммарная цена договоров',
    'Квартиры количество месяц',
    'Квартиры площадь месяц',
    'Квартиры сумма месяц',
    'Нежилые количество месяц',
    'Нежилые площадь месяц',
    'Нежилые сумма месяц',
    'Машино-места количество месяц',
    'Машино-места площадь месяц',
    'Машино-места сумма месяц',
    'Площадь жилых помещений',
    'Площадь нежилых помещений',
    'Площадь жилых и нежилых помещений',
    'Количество жилых помещений',
    'Количество нежилых помещений',
    'Количество машино-мест'

]

# NaN → 0 только в суммируемых колонках
df1[sum_cols] = df1[sum_cols].fillna(0)

# автоматически определяем остальные столбцы
other_cols = [col for col in df1.columns if col not in sum_cols]

# для них берём первое значение
agg_dict = {col: 'first' for col in other_cols}
agg_dict.update({col: 'sum' for col in sum_cols})

df2 = (
    df1
    .groupby(group_cols, as_index=False)
    .agg(agg_dict)
)


In [18]:
cols = [
'Площадь жилых помещений',
'Площадь нежилых помещений',
'Площадь жилых и нежилых помещений',
'Количество жилых помещений',
'Количество нежилых помещений',
'Количество машино-мест'
]

df1[cols] = (
    df1.groupby('Id дом.рф')[cols]
      .transform(lambda x: x.ffill().bfill())
)

In [19]:
cols = [
'Площадь жилых помещений',
'Площадь нежилых помещений',
'Площадь жилых и нежилых помещений',
'Количество жилых помещений',
'Количество нежилых помещений',
'Количество машино-мест'
]

df2[cols] = (
    df2.groupby('Id дом.рф')[cols]
      .transform(lambda x: x.ffill().bfill())
)

In [20]:
# переносим столбцы (cols_to_replace) на страницы с характеристиками и убираем их из других страниц

cols_to_replace = [
    'Площадь жилых помещений',
    'Площадь нежилых помещений',
    'Площадь жилых и нежилых помещений',
    'Количество жилых помещений',
    'Количество нежилых помещений',
    'Количество машино-мест'
]

base_cols = [
    'Название проекта',
    'Корпус',
    'Застройщик',
    'Месяц_год',
    'Id дом.рф',
    '№ проектной декларации'
]

df3 = df1[base_cols + cols_to_replace].copy()
df1 = df1.drop(columns=cols_to_replace)
df4 = df2[base_cols + cols_to_replace].copy()
df2 = df2.drop(columns=cols_to_replace)

In [21]:
# переносим столбцы с продажами месяц к месяцу (cols_to_replace) на отдельные страницы и убираем их из других страниц

cols_to_replace = [
    'Квартиры количество месяц',
    'Квартиры площадь месяц',
    'Квартиры сумма месяц',
    'Нежилые количество месяц',
    'Нежилые площадь месяц',
    'Нежилые сумма месяц',
    'Машино-места количество месяц',
    'Машино-места площадь месяц',
    'Машино-места сумма месяц'
]


base_cols = [
    'Название проекта',
    'Корпус',
    'Застройщик',
    'Месяц_год',
    'Id дом.рф',
    '№ проектной декларации'
]

df5 = df1[base_cols + cols_to_replace].copy()
df1 = df1.drop(columns=cols_to_replace)
df6 = df2[base_cols + cols_to_replace].copy()
df2 = df2.drop(columns=cols_to_replace)

In [22]:
df1 = df1.drop(columns=['№ проектной декларации'], errors='ignore')
df2 = df2.drop(columns=['№ проектной декларации'], errors='ignore')
df2 = df2.drop(columns=['Корпус', 'Id дом.рф'], errors='ignore')
df4 = df4.drop(columns=['Корпус', 'Id дом.рф'], errors='ignore')
df6 = df6.drop(columns=['Корпус', 'Id дом.рф'], errors='ignore')

In [23]:
# определяем порядок столбцов, первыми идут указанные три столбца (first_cols), а дальше по умолчанию
def reorder_columns(df, first_cols):
    first_cols_existing = [col for col in first_cols if col in df.columns]
    other_cols = [col for col in df.columns if col not in first_cols_existing]
    return df[first_cols_existing + other_cols]

first_cols = ['Месяц_год', 'Застройщик', 'Название проекта']

df1 = reorder_columns(df1, first_cols)
df2 = reorder_columns(df2, first_cols)
df3 = reorder_columns(df3, first_cols)
df4 = reorder_columns(df4, first_cols)
df5 = reorder_columns(df5, first_cols)
df6 = reorder_columns(df6, first_cols)

In [24]:
# Сохранение в один Excel файл на разные листы
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    df5.to_excel(writer, sheet_name='Продажи по корпусам мес-мес', index=False)
    df6.to_excel(writer, sheet_name='Продажи по проектам мес-мес', index=False)
    df1.to_excel(writer, sheet_name='Продажи по корпусам исходник', index=False)
    df2.to_excel(writer, sheet_name='Продажи по проектам исходник', index=False)
    df3.to_excel(writer, sheet_name='Характеристики по корпусам', index=False)
    df4.to_excel(writer, sheet_name='Характеристики по проектам', index=False)